<div align="center">

# Convert edf to hdf5

</div>

> These functions are only used here in this notebook, hence why it's not in others

In [1]:
import os
import numpy as np
import mne
import h5py
import re

> Careful with formatting of chb17 as it's named chb17a, chb17b, chb17c instead of just chb17.

In [4]:
def parse_chbmit_summary(summary_path):
    seizures_dict = {}

    with open(summary_path, "r", encoding="latin1") as f:
        text = f.read()

    # split into file sections
    blocks = text.split("File Name:")

    for block in blocks:
        if ".edf" not in block:
            continue

        # ---------------------------------------------------
        # EDF File Name Matching
        # ---------------------------------------------------
        file_match = re.search(r"(chb\d+[a-z]?_\d+\+?\.edf)", block)

        if not file_match:
            file_match = re.search(r"(chb\d+[a-z]?_\d+[^\s]*\.edf)", block)

        if not file_match:
            continue

        file_name = file_match.group(1)

        seizures = []

        # ---------------------------------------------------
        # CASE 1: numbered seizures
        # ---------------------------------------------------
        starts_num = re.findall(r"Seizure\s*\d+\s*Start Time:\s*(\d+)", block)
        ends_num   = re.findall(r"Seizure\s*\d+\s*End Time:\s*(\d+)", block)

        if len(starts_num) > 0 and len(ends_num) > 0:
            seizures = [(int(s), int(e)) for s, e in zip(starts_num, ends_num)]

        else:
            # ---------------------------------------------------
            # CASE 2: unnumbered format
            # ---------------------------------------------------
            start = re.search(r"Seizure Start Time:\s*(\d+)", block)
            end   = re.search(r"Seizure End Time:\s*(\d+)", block)

            if start and end:
                seizures = [(int(start.group(1)), int(end.group(1)))]

        seizures_dict[file_name] = seizures

    return seizures_dict





def convert_dataset(seizure_map, DATASET_PATH, OUTPUT_PATH):
    for file_name, seizures_sec in seizure_map.items():

        edf_path = os.path.join(DATASET_PATH, file_name)
        if not os.path.exists(edf_path):
            print(f"Missing: {edf_path}")
            continue

        print(f"Processing {file_name}")

        # -------------------------
        # Load EDF
        # -------------------------
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw.rename_channels(lambda x: x.strip())

        data = raw.get_data().astype(np.float32)   # (channels, samples)
        fs = float(raw.info["sfreq"])
        ch_names = np.array(raw.ch_names, dtype="S")

        # -------------------------
        # Convert seizures (sec → samples)
        # -------------------------
        seizures_samples = np.array(
            [(int(s * fs), int(e * fs)) for s, e in seizures_sec],
            dtype=np.int64
        )

        if seizures_samples.size == 0:
            seizures_samples = np.zeros((0, 2), dtype=np.int64)

        # -------------------------
        # Save HDF5
        # -------------------------
        h5_path = os.path.join(
            OUTPUT_PATH,
            file_name.replace(".edf", ".h5")
        )

        with h5py.File(h5_path, "w") as f:
            f.create_dataset("signals", data=data, compression="gzip")
            f.create_dataset("sampling_rate", data=fs)
            f.create_dataset("channel_names", data=ch_names)
            f.create_dataset("seizure_intervals", data=seizures_samples)

        print(f"Saved → {h5_path} | seizures: {len(seizures_samples)}")



In [ ]:
nums = [f"{i:02d}" for i in range(24, 25)]

for numero in nums:
    texto = f"../Dataset/chb{numero}/chb{numero}-summary.txt"
    seizure_map = parse_chbmit_summary(texto)
    print(seizure_map)



    DATASET_PATH = f"../Dataset/chb{numero}"
    OUTPUT_PATH = f"../Dataset_hdf5/chb{numero}"

    os.makedirs(OUTPUT_PATH, exist_ok=True)



    # RUN
    convert_dataset(seizure_map, DATASET_PATH,OUTPUT_PATH)



NameError: name 'os' is not defined